In [2]:
import torch
import torch.nn as nn 
import tiktoken 

In [3]:
class MultiHeadAttention(nn.Module): 
  def __init__(self, in_dim, head_dim, context_length, drop_rate, n_heads, qkv_bias=False): 
    super().__init__()
    self.in_dim = in_dim        # width of each token vector: what flows in, and what out_proj projects back to for the residual add
    self.head_dim = head_dim    # width of a single attention head
    self.context_length = context_length
    self.drop_rate = drop_rate
    self.n_heads = n_heads
    self.d_attn = head_dim * n_heads   # total Q/K/V width (all heads concatenated) before out_proj

    self.W_Q = nn.Linear(in_dim, self.d_attn, bias=qkv_bias)                    
    self.W_K = nn.Linear(in_dim, self.d_attn, bias=qkv_bias)
    self.W_V = nn.Linear(in_dim, self.d_attn, bias=qkv_bias)

    # mask fill 
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    # dropout 
    self.dropout = nn.Dropout(drop_rate)

    # project attention output back to in_dim for residual connection
    self.out_proj = nn.Linear(self.d_attn, in_dim)
  
  def forward(self, x):
    batch, num_tokens, _ = x.shape 

    # reshape qkv matrices
    queries = self.W_Q(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
    keys = self.W_K(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
    values = self.W_V(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)

    # attention score 
    attn_score = queries @ keys.transpose(-2, -1)
    scaled_score = attn_score / (self.head_dim ** 0.5)

    # applying causal attention with attention mask 
    scaled_score = scaled_score.masked_fill(self.mask[:num_tokens, :num_tokens], float("-inf"))

    attn_weight = torch.softmax(scaled_score, dim=-1)
    attn_weight = self.dropout(attn_weight)

    # return context vector
    context_vec = (attn_weight @ values).transpose(1,2).contiguous().view(batch, num_tokens, self.d_attn)

    return self.out_proj(context_vec) 

In [4]:
# torch.manual_seed(420)
# x = torch.randn(2, 4, 8)          # b=2, num_tokens=4, in_dim=8
# attn = MultiHeadAttention(in_dim=8, head_dim=4, context_length=1024, n_heads=4, drop_rate=0.1)
# out = attn(x)
# out.shape 

In [5]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
    self.eps = 1e-5

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False) 
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale * norm_x + self.shift 

class FeedForward(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(emb_dim, 4 * emb_dim), 
      nn.GELU(approximate='tanh'),
      nn.Linear(emb_dim * 4, emb_dim), 
    )
  
  def forward(self, x):
    return self.layers(x)

In [6]:
from torch.nn import Dropout

class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.attn = MultiHeadAttention(
      in_dim=cfg["emb_dim"],
      head_dim=cfg["emb_dim"] // cfg["n_heads"],
      context_length=cfg["context_length"],
      drop_rate=cfg["drop_rate"],
      n_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.dropout = Dropout(cfg["drop_rate"])

  def forward(self, x):
    # sub-layer 1
    shortcut = x
    x = self.norm1(x)
    x = self.attn(x)
    x = self.dropout(x)
    x = x + shortcut

    # sub-layer 2
    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.dropout(x)
    x = x + shortcut

    return x

In [7]:
GPT3_CONFIG_125M = {
    "vocab_size": 50257,
    "context_length": 2048,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

In [8]:
class GPT(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.vec_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.layers = nn.Sequential(
      # *, unpacks the list into individual arguments
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )
    self.lnf = LayerNorm(cfg["emb_dim"])
    self.lm_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, x):
    batch_size, seq_len = x.shape
    pos = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
    
    x = self.vec_emb(x) + self.pos_emb(pos)
    x = self.dropout(x)
    x = self.layers(x)
    x = self.lnf(x)
    logits = self.lm_head(x)
    return logits



In [9]:
def generate(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -context_size:]      # crop to context window
      logits = model(idx_cond)               # forward pass
      logits = logits[:, -1, :]             # last position only
      next_token = torch.argmax(logits, dim=-1, keepdim=True)  # greedy
      idx = torch.cat([idx, next_token], dim=1)   # append
    return idx

In [10]:
# load the tokenizer
enc = tiktoken.get_encoding("gpt2")

def create_dataloader(text, batch_size, stride, context_length):
  enc = tiktoken.get_encoding("gpt2") 
  tokens = enc.encode(text) 
  tokens = torch.tensor(tokens)

  # data sampling 
  inputs, targets = [], []
  for i in range(0, len(tokens) - context_length, stride):
    inputs.append(tokens[i:i+context_length])
    targets.append(tokens[i+1:i+context_length+1])
  
  inputs = torch.stack(inputs)
  targets = torch.stack(targets)

  # batches 
  for i in range(0, inputs.size(0) - batch_size + 1, batch_size):
    x = inputs[i:i+batch_size]
    y = targets[i:i+batch_size]
    yield x, y

In [11]:
import torch.nn.functional as F

def train(model, text, num_epochs, batch_size, context_length, lr):
  device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
  model = model.to(device)

  optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

  step = 0
  for epoch in range(num_epochs):
      model.train()
      for x, y in create_dataloader(text, batch_size, stride=context_length, context_length=context_length):
          x, y = x.to(device), y.to(device)
          # zero grads
          optimizer.zero_grad()

          # forward pass → logits
          logits = model(x)

          # compute loss
          loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))

          # backward
          loss.backward()

          # optimizer step
          optimizer.step()

          # print loss
          step += 1
          if step == 1 or step % 10 == 0:
            print(f"Epoch {epoch}, Step {step}, Loss: {loss.item():.4f}")
        
  return model


In [12]:
torch.manual_seed(123)
model = GPT(GPT3_CONFIG_125M)

In [13]:
with open("../text/the-verdict.txt", "r") as f:  text = f.read()
train(model, text, num_epochs=20, batch_size=4, context_length=512, lr=3e-4)

Epoch 0, Step 1, Loss: 11.0108


KeyboardInterrupt: 

In [ ]:
# Save the trained weights (run once, after training finishes)
torch.save(model.state_dict(), "gpt3_verdict.pth")
print("saved -> gpt3_verdict.pth")

saved -> gpt3_verdict.pth


In [ ]:
enc = tiktoken.get_encoding("gpt2")
prompt = "The sky is blue"
tokens = enc.encode(prompt)
idx = torch.tensor(tokens).unsqueeze(0).to(next(model.parameters()).device)

# to generate text
model.eval() 
with torch.no_grad():
    out = generate(model, idx, max_new_tokens=50, context_size=256)

decoded = enc.decode(out.squeeze(0).tolist())
print(decoded)

NameError: name 'tiktoken' is not defined